# LLM Runtime Diagnostics

This notebook reads persisted `llm_metrics` rows from `storage/runs.db` for long-term tuning.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('storage/runs.db')
if not DB_PATH.exists():
    raise FileNotFoundError(f'Missing DB: {DB_PATH}')

with sqlite3.connect(DB_PATH) as conn:
    table_names = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()}
    if 'llm_metrics' not in table_names:
        df = pd.DataFrame()
    else:
        df = pd.read_sql_query(
            """
            SELECT created_at AS timestamp, status, category, mode, think, model, endpoint,
                   attempts, elapsed_s, done_reason, eval_count, prompt_eval_count,
                   tokens_per_sec, error_count, retry_after_s
            FROM llm_metrics
            ORDER BY created_at ASC, id ASC
            """,
            conn,
        )

if not df.empty:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['elapsed_s'] = pd.to_numeric(df['elapsed_s'], errors='coerce')
    df['tokens_per_sec'] = pd.to_numeric(df['tokens_per_sec'], errors='coerce')

print(f'Rows loaded: {len(df)}')
df.tail(10)


In [ ]:
if df.empty:
    print('No llm_metrics rows found yet.')
else:
    summary = (
        df.groupby(['status', 'category', 'mode', 'think', 'endpoint'], dropna=False)
          .agg(
              calls=('status', 'size'),
              elapsed_avg_s=('elapsed_s', 'mean'),
              elapsed_p95_s=('elapsed_s', lambda s: s.quantile(0.95)),
              tps_avg=('tokens_per_sec', 'mean'),
          )
          .reset_index()
          .sort_values(['status', 'calls'], ascending=[True, False])
    )
    summary


In [ ]:
if df.empty:
    print('No data to plot')
else:
    import matplotlib.pyplot as plt

    ok = df[df['status'] == 'ok'].copy()
    if ok.empty:
        print('No successful calls to plot')
    else:
        ok = ok.sort_values('timestamp')
        ok['series'] = ok['endpoint'].astype(str) + '|think=' + ok['think'].astype(str)
        plt.figure(figsize=(12, 4))
        for series, part in ok.groupby('series'):
            plt.plot(part['timestamp'], part['elapsed_s'], marker='o', linewidth=1, label=series)
        plt.legend(loc='upper right')
        plt.title('LLM Call Latency (seconds)')
        plt.ylabel('elapsed_s')
        plt.xlabel('timestamp')
        plt.grid(True, alpha=0.25)
        plt.tight_layout()
        plt.show()
